# 3. Unsupervised Clustering: Customer Behavior
This notebook implements k-Means clustering, determines the optimal number of clusters, and analyzes cluster centroids for business insights.

## 1. Import Required Libraries
We import all necessary libraries for clustering, visualization, and analysis.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

## 2. Load Data and Preprocessing Pipeline
We load the cleaned dataset and the preprocessing pipeline from previous steps.

In [ ]:
# Load cleaned data (repeat cleaning steps for reproducibility)
df = pd.read_csv('https://huggingface.co/datasets/millat/e-commerce-orders/resolve/main/ecommerce_orders.csv')
drop_cols = ['order_id', 'customer_id', 'product_id', 'shipping_address', 'billing_address']
df_clean = df.drop(columns=drop_cols)
df_clean['order_date'] = pd.to_datetime(df_clean['order_date'])
df_clean['shipping_date'] = pd.to_datetime(df_clean['shipping_date'])
for col in ['order_date', 'shipping_date']:
    df_clean[f'{col}_year'] = df_clean[col].dt.year
    df_clean[f'{col}_month'] = df_clean[col].dt.month
    df_clean[f'{col}_day'] = df_clean[col].dt.day
    df_clean[f'{col}_hour'] = df_clean[col].dt.hour
df_clean['days_to_ship'] = (df_clean['shipping_date'] - df_clean['order_date']).dt.days
df_clean = df_clean.drop(columns=['order_date', 'shipping_date'])
df_clean = df_clean.fillna({'days_to_ship': df_clean['days_to_ship'].median()})
df_clean = df_clean.dropna()

# Load preprocessing pipeline
preprocessor = joblib.load('../artifacts/preprocessing_pipeline.joblib')

## 3. Prepare Features for Clustering
We select and preprocess features for k-means clustering.

In [ ]:
# Exclude target columns for unsupervised learning
X_cluster = df_clean.drop(columns=['customer_segment', 'delivery_status', 'price'])
X_cluster_processed = preprocessor.transform(X_cluster)
X_cluster_processed.shape

## 4. Select Number of Clusters (k)
We use the Elbow Method and Silhouette Score to determine the optimal number of clusters.

In [ ]:
# Elbow method and silhouette score
wcss = []
silhouette_scores = []
k_range = range(2, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_cluster_processed)
    wcss.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_cluster_processed, kmeans.labels_))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(k_range, wcss, marker='o')
plt.title('Elbow Method (WCSS)')
plt.xlabel('Number of clusters (k)')
plt.ylabel('WCSS')
plt.subplot(1, 2, 2)
plt.plot(k_range, silhouette_scores, marker='o', color='orange')
plt.title('Silhouette Score')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette Score')
plt.tight_layout()
plt.show()

## 5. Apply k-Means Clustering
We fit k-means with the selected number of clusters and analyze the results.

In [ ]:
# Choose k based on elbow/silhouette (example: k=4)
k = 4
kmeans = KMeans(n_clusters=k, random_state=42)
clusters = kmeans.fit_predict(X_cluster_processed)
df_clean['cluster'] = clusters

# Cluster sizes
print(df_clean['cluster'].value_counts())

## 6. Cluster Interpretation
We analyze the centroids and feature distributions for each cluster to extract business insights.

In [ ]:
# Analyze cluster centroids (in original feature space is complex due to encoding)
# Instead, analyze summary statistics for each cluster
for c in range(k):
    print(f"\nCluster {c} summary:")
    display(df_clean[df_clean['cluster'] == c].describe(include='all'))

## 7. Business Insights from Clusters
- Each cluster represents a unique customer segment or behavior pattern.
- Use these insights for targeted marketing, personalized promotions, dynamic pricing, and inventory planning.